# 7. 现代卷积神经网络

在本章中的每一个模型都曾一度占据主导地位，其中许多模型都是ImageNet竞赛的优胜者。

> ImageNet竞赛自2010年以来，一直是计算机视觉中监督学习进展的指向标。

这些模型包括：

*   **AlexNet**。它是第一个在大规模视觉竞赛中击败传统计算机视觉模型的大型神经网络；

*   **使用重复块的网络（VGG）**。它利用许多重复的神经网络块；

*   **网络中的网络（NiN）**。它重复使用由卷积层和 $1 \times 1$ 卷积层（用来代替全连接层）来构建深层网络;

*   **含并行连结的网络（GoogLeNet）**。它使用并行连结的网络，通过不同窗口大小的卷积层和最大汇聚层来并行抽取信息；

*   **残差网络（ResNet）**。它通过残差块构建跨层的数据通道，是计算机视觉中最流行的体系架构；

*   **稠密连接网络（DenseNet）**。它的计算成本很高，但给我们带来了更好的效果。

虽然深度神经网络的概念非常简单——将神经网络堆叠在一起。但由于不同的网络架构和超参数选择，这些神经网络的性能会发生很大变化。本章介绍的神经网络是将人类直觉和相关数学见解结合后，经过大量研究试错后的结晶。 

---


## 7.1 深度卷积神经网络 (AlexNet)

> 如果说 LeNet 是神经网络的“童年”，那么 **AlexNet** 就是神经网络的“成年礼”。2012 年，AlexNet 在 ImageNet 竞赛中以绝对优势夺冠，彻底终结了传统计算机视觉的统治，开启了深度学习的黄金时代。

### 1. 核心动机：为什么 LeNet 不够了？

LeNet 在处理手写数字（28x28）时非常完美，但在面对真实世界的自然图像（如 ImageNet 的高清图）时遇到了瓶颈：

1.  **特征复杂度**：真实图像的纹理、光照和背景极其复杂，LeNet 的参数量不足以捕捉这些特征。
2.  **计算资源**：2012 年左右，GPU 算力开始爆发，使得训练更深、更宽的网络成为可能。
3.  **激活函数限制**：Sigmoid 在深层网络中极易导致**梯度消失**。

---

### 2. AlexNet 的五大核心创新

1.  **更深的网络结构**：8 层加权层（5 层卷积 + 3 层全连接）。
2.  **使用 ReLU 激活函数**：用 $\text{max}(0, x)$ 代替 Sigmoid。ReLU 在正区间导数为 1，彻底解决了深层网络的梯度消失问题，且计算速度极快。
3.  **使用 Dropout (暂退法)**：在最后的全连接层引入 Dropout（我们在 4.6 节学过），有效控制了巨大参数量带来的过拟合。
4.  **数据增广 (Data Augmentation)**：通过随机裁剪、翻转和亮度调整，人为增加训练样本，提高模型鲁棒性。
5.  **最大汇聚 (Max Pooling)**：用最大汇聚代替 LeNet 的平均汇聚，更能突出图像的显著特征。

---

### 3. 构建 AlexNet

AlexNet 的输入通常是 $224 \times 224$。由于 Fashion-MNIST 只有 $28 \times 28$，我们在加载数据时需要将其**强制放大**。

```python
import torch
from torch import nn, Tensor

def get_alexnet_model(num_classes: int = 10) -> nn.Sequential:
    """构建 AlexNet 模型。
    
    结构参考 2012 年经典论文，针对 Fashion-MNIST 输出 10 类。
    """
    net = nn.Sequential(
        # 第一块：大尺寸卷积核提取粗略特征
        # 输入 (1, 224, 224) -> 输出 (64, 54, 54)
        nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=1), nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),
        
        # 第二块：减小核尺寸，增加通道数
        # 输出 (192, 26, 26)
        nn.Conv2d(64, 192, kernel_size=5, padding=2), nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),
        
        # 第三块：连续三个卷积层提取精细特征
        nn.Conv2d(192, 384, kernel_size=3, padding=1), nn.ReLU(),
        nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),
        nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),
        
        # 第四块：全连接层 + Dropout
        nn.Flatten(),
        # AlexNet 的 FC 层非常庞大，参数量主要集中在这里
        nn.Linear(256 * 6 * 6, 4096), nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(4096, 4096), nn.ReLU(),
        nn.Dropout(p=0.5),
        
        # 输出层
        nn.Linear(4096, num_classes)
    )
    return net

# --- 验证维度流转 ---
X = torch.randn(1, 1, 224, 224)
net = get_alexnet_model()
for layer in net:
    X = layer(X)
    print(f"{layer.__class__.__name__:15} 输出形状: {X.shape}")
```

---

## 4. 关键工程配置：数据预处理

为了适配 AlexNet，我们需要修改之前的 `load_data_fashion_mnist` 函数，加入 `Resize` 步骤。

```python
import d2l_utils as d2l # 假设你之前的工具包

def load_data_alexnet(batch_size: int, resize: int = 224) -> tuple:
    """加载 Fashion-MNIST 并调整尺寸以适配 AlexNet。"""
    # 这里调用 3.5 节重构后的 load_data 函数
    return d2l.load_data_fashion_mnist(batch_size, resize=resize)

# 建议配置
batch_size = 128
train_iter, test_iter = load_data_alexnet(batch_size)
```

---

## 5. 细致梳理：AlexNet 的数学与逻辑

1.  **计算量与参数量**：
    *   AlexNet 虽然只有 8 层，但参数量高达 **6000 万**个。
    *   绝大部分参数集中在第一个全连接层：$256 \times 6 \times 6 \times 4096 \approx 3700$ 万。
2.  **ReLU 的胜利**：
    *   ReLU 的计算就是一行 `if x > 0`，而 Sigmoid 涉及复杂的指数运算。这使得 AlexNet 的训练速度比同规模的 Sigmoid 网络快 6 倍以上。
3.  **为什么第一层用 11x11 的大核？**
    *   在输入图片很大（224x224）时，需要一个足够大的窗口来捕捉最初的物体轮廓。

---

## 6. 关键工程暗知识 (Engineering Wisdom)

*   **显存预警**：AlexNet 的全连接层非常吃显存。如果你的显卡显存较小（如 4GB），请务必调小 `batch_size`（如 64 或 32）。
*   **学习率调优**：由于 AlexNet 比较深，建议初始学习率设小一点（如 0.01），并配合 Adam 优化器。

---

## 7. 总结 Checklist

*   [x] **ReLU**：明白它为什么能取代 Sigmoid（解决梯度消失）。
*   [x] **Dropout**：明白它在 AlexNet 这种大参数模型中的必要性。
*   [x] **Resize**：掌握如何处理输入尺寸不匹配的问题。
*   [x] **架构感**：记住 AlexNet “卷积-卷积-卷积卷积卷积-全连接”的节奏。

---

### 💡 你的实战挑战：

1.  **参数规模分析**：编写一段代码，计算 `get_alexnet_model` 中每一层参数的具体数量，并打印出占比最大的那一层。
2.  **训练实验**：在 GPU 上运行 AlexNet 训练。观察在前 3 个 Epoch 中，AlexNet 的准确率提升速度是否明显快于之前的 LeNet？

**AlexNet 的“暴力美学”你感受到了吗？如果理顺了，我们就进入 7.2 节：使用块的网络 (VGG)。我们将学习如何通过重复使用简单的“卷积块”来构建更深、更规律的网络！**
